## Leetcode 180. Consecutive Numbers

In [0]:
data = [(1,1),(2,1),(3,1),(4,2),(5,1),(6,2),(7,2)]
Logs = spark.createDataFrame(data, ['id','num'])
Logs.createOrReplaceTempView('Logs')


Logs
| id | num |
|----|-----|
| 1  | 1   |
| 2  | 1   |
| 3  | 1   |
| 4  | 2   |
| 5  | 1   |
| 6  | 2   |
| 7  | 2   |

### Spark SQL

In [0]:
spark.sql('''
          select distinct l1.num as ConsecutiveNums from Logs l1 join Logs l2 on l1.id+1= l2.id join Logs l3 on l2.id+1=l3.id
          where (l1.num=l2.num) and (l2.num=l3.num)
          order by ConsecutiveNums
          ''').show()

### Spark DataFrame API

In [0]:
from pyspark.sql.functions import lag, lead, col, countDistinct
from pyspark.sql.window import Window

spark.conf.set("spark.sql.shuffle.partitions","1")

w = Window.orderBy(col('id'))
Logs.withColumn('prev', lag('num').over(w))\
    .withColumn('nxt', lead('num').over(w))\
    .filter((col('num')==col('prev')) & (col('num')==col('nxt')))\
        .select(col("num").alias("ConsecutiveNums")).distinct().show()